# CLIFFGUARD — Google Colab (T4-ready)

Runs Fold A calibration and Fold B cliff measurement on a free T4 GPU.  
Every scheme checkpoints to Drive after it finishes; a disconnected session loses at most one scheme.

**How to run:** Execute cells top-to-bottom.  
Cell C5 auto-restarts the kernel once (numpy pin). After the restart, re-run all cells from C1 — the restart check will pass and execution continues normally.

**What you get:**
- `fold_a/calibration_summary.json` — refusal direction r̂ + per-scheme threshold τ_q
- `fold_b/cliff_results.json` — Δ_cliff, Δ_W-cliff, Δ_B-cliff, H1 verdict
- Persistent results at `/content/drive/MyDrive/cliffguard/results/<run_id>/`

In [ ]:
# C1 — GPU check
# If VRAM shows 0 GB, switch via Runtime > Change runtime type > T4 GPU and re-run.
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
else:
    raise RuntimeError("No GPU detected. Switch to T4 via Runtime > Change runtime type.")

In [ ]:
# C2 — Mount Google Drive and create persistent directories
from google.colab import drive
drive.mount('/content/drive')

import os
for _d in [
    '/content/drive/MyDrive/cliffguard/results',
    '/content/drive/MyDrive/cliffguard/datasets/folds/fold_a',
    '/content/drive/MyDrive/cliffguard/datasets/folds/fold_b',
]:
    os.makedirs(_d, exist_ok=True)
print("Drive mounted. Persistent directories ready.")

In [ ]:
# C3 — Clone or update the CLIFFGUARD repo
%cd /content
!git clone https://github.com/parnish007/CLIFFGUARD.git 2>/dev/null || (cd CLIFFGUARD && git pull)
%cd /content/CLIFFGUARD
!git log -1 --oneline

In [ ]:
# C4 — Pin numpy < 2 (required to avoid NF4 weight-conversion crash on Llama-3.2)
#
# Colab ships with numpy 2.x. bitsandbytes + Llama-3.2 fail with
# "ModuleNotFoundError: No module named 'numpy.rec'" under numpy 2.x.
# This cell detects the version, downgrades if needed, then force-restarts
# the kernel so the old numpy is fully evicted from memory.
# After restart, re-run ALL cells from C1 — this check will pass cleanly.
import sys, subprocess

try:
    import numpy as _np
    _major = int(_np.__version__.split('.')[0])
    print(f"numpy {_np.__version__} detected.")
except ImportError:
    _major = -1
    print("numpy not installed.")

if _major >= 2:
    print("Downgrading numpy to <2 and restarting kernel...")
    print("After restart, re-run all cells from C1.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2"], check=True)
    import os, signal
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print(f"numpy OK (< 2). No restart needed.")

In [ ]:
# C5 — Install all dependencies
# Run order matters: numpy<2 is installed first (already done in C4),
# then transformers and bitsandbytes pinned to versions tested on T4 + Llama-3.2.
%cd /content/CLIFFGUARD
!pip install -q "numpy<2"
!pip install -q -e /content/CLIFFGUARD
!pip install -q \
    "torch" \
    "transformers>=4.45,<4.50" \
    "bitsandbytes>=0.43,<0.45" \
    "accelerate" \
    "sentencepiece" \
    "protobuf" \
    "datasets" \
    "huggingface_hub"
print("\nAll packages installed.")
import numpy; print(f"numpy version: {numpy.__version__}")

In [ ]:
# C6 — Install llama-cpp-python (needed for GGUF schemes; optional for FP16/NF4 on T4)
# Tries the prebuilt CUDA 12.2 wheel first (~5 sec).
# Falls back to source build (~8-10 min) if the wheel is unavailable for your CUDA/Python version.
!pip install -q llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
  || CMAKE_ARGS="-DLLAMA_CUDA=on" pip install --force-reinstall --no-cache-dir -q llama-cpp-python
print("llama-cpp-python installed.")

## HuggingFace Authentication

Your Colab secret should be named **`cliffguard_test`**. Cell C7 reads it automatically.

If you haven't added it yet:
1. Click the **key icon** in the left sidebar
2. Add secret — Name: **`cliffguard_test`**, Value: your HF read token
3. Toggle **Notebook access** ON
4. Accept the model license gate (one-time, on HuggingFace website) for each model you want to use:
   - [Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
   - [Llama-3.2-1B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct)

In [ ]:
# C7 — HuggingFace login
# Tries 'cliffguard_test' secret first, then 'HF_TOKEN', then interactive prompt.
from huggingface_hub import login

_token = None
try:
    from google.colab import userdata
    for _secret_name in ('cliffguard_test', 'HF_TOKEN'):
        try:
            _t = userdata.get(_secret_name)
            if _t:
                _token = _t
                print(f"[hf] using Colab secret '{_secret_name}'")
                break
        except Exception:
            continue
except ImportError:
    pass

if _token:
    login(token=_token, add_to_git_credential=False)
    print("[hf] authenticated.")
else:
    print("[hf] no Colab secret found. Paste your HF token below:")
    login()

In [ ]:
# C8 — Import colab_helper and print status banner
import sys
sys.path.insert(0, '/content/CLIFFGUARD/notebooks')
import colab_helper as ch
ch.ensure_repo_cwd()
ch.banner()

In [ ]:
# C9 — Smoke test (no GPU, no datasets, ~1 sec)
# Exercises the full pipeline shape with synthetic arrays.
# If this fails, stop here and check the install (C5).
ch.ensure_repo_cwd()
!python scripts/dry_run.py --tier A --scheme FP16
!python scripts/dry_run.py --tier C --scheme GGUF_Q3_K_M
print("Smoke test passed.")

In [ ]:
# C10 — Link Drive datasets into repo + download Fold A corpus
# First-run download: ~5-10 min for 500 samples.
# Subsequent sessions skip the download (data/ is a symlink into Drive).
ch.ensure_repo_cwd()
ch.symlink_datasets_from_drive()
!python scripts/download_fold_a.py --download --max 500 --target-dir data/folds/fold_a
!ls -lh data/folds/fold_a/

In [ ]:
# C11 — Auto-select model and schemes based on free VRAM
# T4 (14-15 GB free): Llama-3.2-3B-Instruct, schemes=[FP16, NF4]
# L4  (22-23 GB free): Llama-3.1-8B-Instruct, schemes=[FP16, NF4]
# A100 (35+ GB free): Llama-3.1-8B-Instruct, schemes=[FP16, NF4, AWQ_INT4]
ch.ensure_repo_cwd()
config = ch.choose_model()
print(config)

# To override, uncomment and edit:
# config['model_id'] = 'meta-llama/Llama-3.2-1B-Instruct'
# config['layer']    = 8
# config['schemes']  = ['FP16', 'NF4']

## Fold A — Calibration

`run_fold_a_with_checkpoint` runs Arditi diff-in-means calibration per scheme.

- Checkpoints after every `(model, scheme)` pair — re-run the cell after a disconnect to resume.
- Calls `torch_cleanup()` between schemes so NF4 loads without VRAM residue from FP16.
- **NF4 loading takes 3-5 minutes on T4. Do not stop the cell — let it run.**
- Syncs to Drive after every scheme.

In [ ]:
# C12 — Run Fold A with checkpointing
# NF4 loading takes 3-5 minutes. Do not interrupt.
# If the cell was stopped mid-scheme, simply re-run it.
ch.ensure_repo_cwd()
ch.run_fold_a_with_checkpoint(config)

In [ ]:
# C13 — Manual Drive sync (optional)
# run_fold_a_with_checkpoint already syncs after each scheme.
# Use this cell only if you need an extra sync.
ch.ensure_repo_cwd()
ch.sync_artifacts_to_drive()

In [ ]:
# C14 — Download Fold B corpus (AdvBench + JailbreakBench)
# Idempotent: skips if both JSONL files already exist.
ch.ensure_repo_cwd()
ch.assemble_fold_b()

In [ ]:
# C15 — Fold isolation audit
# Asserts that no prompt appears in both Fold A and Fold B (SHA-256 disjointness).
# Required before Fold B to avoid data contamination.
ch.ensure_repo_cwd()
ch.fold_isolation_audit()

In [ ]:
# C16 — Verify Fold A is complete before Fold B
# Raises RuntimeError if any requested scheme is missing from the checkpoint.
# This prevents the silent cliff(FP16, FP16)=0 result that broke earlier runs.
ch.ensure_repo_cwd()
ch.verify_fold_a_complete(config)

## Fold B — Cliff Measurement

Reads saved `r_hat_*.npz` from `fold_a/` (no re-calibration).

Computes per scheme (vs FP16 baseline):
- `Δ_cliff` — geometric distance between refusal directions
- `Δ_W-cliff` — Wasserstein distance between margin distributions
- `Δ_B-cliff` — PROBE-RM margin proxy (Phase C will use StrongREJECT + Llama-Guard)

Output: `fold_b/cliff_results.json` with `cliff_boundary` and `h1_accepted_for_this_family`.

In [ ]:
# C17 — Run Fold B and sync results to Drive
ch.ensure_repo_cwd()
ch.run_fold_b_with_checkpoint(config)
ch.sync_artifacts_to_drive()

In [ ]:
# C18 — Inspect results
ch.ensure_repo_cwd()
import json, pathlib
from cliffguard.eval.results_writer import list_runs

for _run in list_runs(pathlib.Path('artifacts')):
    print('=' * 60)
    print('Run:', _run.name)
    for _fname in [
        'fold_a/calibration_summary.json',
        'fold_a/checkpoint.json',
        'fold_b/cliff_results.json',
        'fold_b/checkpoint.json',
    ]:
        _p = _run / _fname
        if _p.exists():
            print(f'\n--- {_fname} ---')
            print(json.dumps(json.load(open(_p)), indent=2))

## Done

**Results location:** `/content/drive/MyDrive/cliffguard/results/<run_id>/`  
This path survives session disconnects.

**To download locally:**
```
Files panel -> that path -> right-click -> Download
```
or:
```python
!zip -r /tmp/cliffguard_results.zip /content/drive/MyDrive/cliffguard/results/
from google.colab import files
files.download('/tmp/cliffguard_results.zip')
```

**Resuming a killed session:**
1. Reconnect to a T4 runtime
2. Re-run cells C1 through C8 (Drive mount, clone, numpy check, install, login, helper import)
3. Re-run C11 (choose_model)
4. Re-run C12 (Fold A) — checkpoint skips already-completed schemes
5. Re-run C17 (Fold B) — picks up from checkpoint

**Known limitations (not bugs):**
- `Δ_B-cliff` in Phase B is a PROBE-RM margin proxy, not StrongREJECT + Llama-Guard (Phase C work)
- Fold B corpus is JBB + AdvBench only (HarmBench and ArtPrompt absent)
- AWQ scheme is Linux-only and is skipped automatically on T4 if added to `config['schemes']`